In [1]:
import sys
import os
from pathlib import Path
from dotenv import load_dotenv
from google import genai
from google.genai import types
from PIL import Image
import base64
import io
import json
from datetime import datetime
import uuid
import time
import re

import brand_director as bd
import nametag_prompts as pn

bd.setup_environment()

# 함수에서 반환된 키를 변수에 저장하여 다른 곳에서 활용
GEMINI_API_KEY, GEMINI_IMAGE_API_KEY = bd.load_gemini_keys()


# 시스템 프롬프트를 변수에 저장하여 다른 곳에서 활용
SYSTEM_PROMPT = pn.SYSTEM_PROMPT

✅ Gemini API 키 로드 완료
✅ Gemini Image API 키 로드 완료


In [2]:
brand_data = bd.get_brand_info()
brand_data


🎯 NameTag 브랜드 가이드 생성

📝 브랜드 정보를 입력해주세요:


✅ 입력된 브랜드 정보:
업종/서비스: 초기 창업자를 위한 브랜딩 코파일럿 AI 툴을 가진 플렛폼
브랜드 감성: 모던, 신뢰, 기히학적, 기계적
타겟 고객: 대학교를 다니고 있는 초기 창업 팀, 사업을 처음 시작 하는 1인 사업자 또는 소상공인
추가 키워드: AI, 코파일럿, 도움을 주는, 브랜딩, 브랜드, 트랜디한, 초기창업자, 학생 창업팀


{'business_type': '초기 창업자를 위한 브랜딩 코파일럿 AI 툴을 가진 플렛폼',
 'vibes': ['모던', '신뢰', '기히학적', '기계적'],
 'target': '대학교를 다니고 있는 초기 창업 팀, 사업을 처음 시작 하는 1인 사업자 또는 소상공인',
 'keywords': 'AI, 코파일럿, 도움을 주는, 브랜딩, 브랜드, 트랜디한, 초기창업자, 학생 창업팀'}

In [3]:
raw_prompt_O = pn.get_O_interview_prompt()
prompt_O = bd.get_prompt(raw_prompt_O, brand_data)


📨 Gemini API에 요청할 프롬프트:

[기본 입력 정보]
- 업종/서비스: 초기 창업자를 위한 브랜딩 코파일럿 AI 툴을 가진 플렛폼
- 브랜드 감성: 모던, 신뢰, 기히학적, 기계적
- 타겟 고객: 대학교를 다니고 있는 초기 창업 팀, 사업을 처음 시작 하는 1인 사업자 또는 소상공인
- 확정된 브랜드명: 브랜드명 미정

창업자가 브랜드의 초기 아이디어를 입력했습니다.
당신의 다음 임무는 이 답변들을 바탕으로, 서로 완전히 다른 4개의 [브랜드 초안 후보(MVB)]를 생성하는 것입니다.

[최종적으로 당신이 다음 단계에서 생성해야 할 4가지 초안 항목]
1. 브랜드명 및 의미: 타겟에 맞는 직관적이거나 감각적인 네이밍
2. 핵심 슬로건: 브랜드의 차별점을 보여주는 15자 이내의 한 줄 카피
3. 탄생 서사 요약: 타겟의 페인포인트(Pain-point)를 해결하는 100자 이내의 스토리
4. 시드 컬러(Seed Color): 브랜드의 감성을 대변하는 핵심 색상 (Hex Code)과 그 이유

위의 4가지 항목을 매력적이고 구체적으로 도출하기 위해, 현재 창업자의 입력값에서 '가장 비어있는 구체적인 디테일'을 파악하여 물어볼 핵심 질문 n개를 생성하세요.

[질문 개수(n) 결정 및 질문 생성 가이드라인]

1. 타겟 고객(Target)의 구체성 평가
- 모호할 때: "2030 여성", "학생"처럼 인구통계학적으로만 뭉뚱그려져 있다면, 타겟의 라이프스타일이나 페인포인트(Pain-point)를 좁히는 질문을 1개 추가하세요.
- 명확할 때: "자기 계발에 돈을 아끼지 않는 3년 차 직장인"처럼 구체적이라면 타겟 관련 질문은 생략하세요.

2. 브랜드 감성(Vibes)의 일관성 및 시각화 평가
- 모호할 때: "따뜻한, 좋은, 예쁜" 등 시각적으로 상상하기 어렵거나, "모던한, 레트로한"처럼 충돌하는 키워드가 섞여 있다면, 브랜드의 인격(Persona)이나 구체적인 시각적 메타포를 고르게 하는 질문을 1개 추가하세요.
- 명확할 때: "새벽 안개 낀 숲",

In [4]:
raw_text_O = bd.request_gemini_api(prompt_O, SYSTEM_PROMPT, GEMINI_API_KEY=GEMINI_API_KEY)
parsed_response_O = bd.parse_ai_response(raw_text_O)


----------------------------------------------------------------------------------------------------
💬 AI에 요청 중입니다... 잠시만 기다려주세요.
----------------------------------------------------------------------------------------------------

✅ AI 응답 수신 완료!
응답 텍스트: {
  "required_question_count": 3,
  "reasoning": "타겟 고객의 범위가 대학생부터 소상공인까지 넓어 이들이 공통적으로 느끼는 '가장 큰 결핍'...
✅ JSON 파싱 완료!
파싱된 JSON: {"required_question_count": 3, "reasoning": "\ud0c0\uac9f \uace0\uac1d\uc758 \ubc94\uc704\uac00 \ub300\ud559\uc0dd\ubd80\ud130 \uc18c\uc0c1\uacf5\uc778\uae4c\uc9c0 \ub113\uc5b4 \uc774\ub4e4\uc774 \uacf5\ud1b5\uc801\uc73c\ub85c \ub290\ub07c\ub294 '\uac00\uc7a5 \ud070 \uacb0\ud54d'\uc744 \ud30c\uc545\ud574\uc57c \ud558\uba70, AI \ud234\ub85c\uc11c \uc81c\uacf5\ud560 '\ud575\uc2ec \uac00\uce58'\uc758 \uacb0\uc774 \uc790\ub3d9\ud654\uc778\uc9c0 \uac00\uc774\ub4dc\uc778\uc9c0 \uad6c\uccb4\ud654\uac00 \ud544\uc694\ud569\ub2c8\ub2e4. \ub610\ud55c '\uae30\uacc4\uc801/\uae30\ud558\ud559\uc801' \uac10\uc131\uc774 \ucc2

In [5]:
# AI가 왜 이런 질문을 만들었는지 사용자에게 보여주면 신뢰도가 확 올라갑니다!
print(f"💡 AI 분석: {parsed_response_O['reasoning']}\n")

# 사용자의 최종 답변을 모아둘 리스트
collected_answers = bd.conduct_ai_interview(parsed_response_O, title="🤖 초기 브랜드 방향성 진단 인터뷰")
interview_data = bd.format_interview_responses(collected_answers)

💡 AI 분석: 타겟 고객의 범위가 대학생부터 소상공인까지 넓어 이들이 공통적으로 느끼는 '가장 큰 결핍'을 파악해야 하며, AI 툴로서 제공할 '핵심 가치'의 결이 자동화인지 가이드인지 구체화가 필요합니다. 또한 '기계적/기하학적' 감성이 차갑고 딱딱한 느낌일지, 정교하고 신뢰감 있는 느낌일지 인격(Persona)을 좁히기 위해 3가지 질문을 생성했습니다.


🤖 초기 브랜드 방향성 진단 인터뷰
💡 AI 분석: 타겟 고객의 범위가 대학생부터 소상공인까지 넓어 이들이 공통적으로 느끼는 '가장 큰 결핍'을 파악해야 하며, AI 툴로서 제공할 '핵심 가치'의 결이 자동화인지 가이드인지 구체화가 필요합니다. 또한 '기계적/기하학적' 감성이 차갑고 딱딱한 느낌일지, 정교하고 신뢰감 있는 느낌일지 인격(Persona)을 좁히기 위해 3가지 질문을 생성했습니다.


[질문 1/3. 이 서비스가 타겟 고객의 페인포인트 중 가장 우선적으로 해결해 주어야 할 가치는 무엇인가요?]
  1. 막막함 해소: 브랜드의 방향성을 잡지 못하는 초보자를 위한 단계별 가이드
  2. 비용과 시간 절감: 전문가 고용 없이 몇 분 만에 결과물을 만드는 압도적 효율
  3. 심미적 완성도: 디자인 감각이 없어도 누구나 만들 수 있는 수준 높은 비주얼
  4. 논리적 근거: 내 브랜드가 왜 이렇게 디자인되어야 하는지에 대한 데이터 기반 설득력

[질문 2/3. 언급하신 '기하학적, 기계적' 감성이 브랜드에서 어떤 이미지로 투영되길 원하시나요?]
  1. 정교함: 오차 없이 완벽하게 짜인 설계도와 같은 신뢰감
  2. 미래지향: 기존의 관습을 깨는 혁신적이고 실험적인 느낌
  3. 미니멀리즘: 불필요한 장식은 빼고 핵심 기능에만 집중한 간결함
  4. 구조적 안점감: 단단한 블록이 쌓여 만들어진 듯한 튼튼한 기초

[질문 3/3. 이 AI 코파일럿은 사용자에게 어떤 '파트너'로 인식되길 바라시나요?]
  1. 냉철한 분석가: 데이터와 논리로 브랜드를 진단하는 AI 전문가
  2. 친절한 튜터: 창업

In [6]:
raw_prompt_1 = pn.get_O_mvb_prompt(interview_data)
prompt_1 = bd.get_prompt(raw_prompt_1, brand_data)


📨 Gemini API에 요청할 프롬프트:

[기본 입력 정보]
- 업종/서비스: 초기 창업자를 위한 브랜딩 코파일럿 AI 툴을 가진 플렛폼
- 브랜드 감성: 모던, 신뢰, 기히학적, 기계적
- 타겟 고객: 대학교를 다니고 있는 초기 창업 팀, 사업을 처음 시작 하는 1인 사업자 또는 소상공인
- 확정된 브랜드명: 브랜드명 미정

[심층 인터뷰 결과]
Q1. 이 서비스가 타겟 고객의 페인포인트 중 가장 우선적으로 해결해 주어야 할 가치는 무엇인가요?
A: 논리적 근거: 내 브랜드가 왜 이렇게 디자인되어야 하는지에 대한 데이터 기반 설득력

Q2. 언급하신 '기하학적, 기계적' 감성이 브랜드에서 어떤 이미지로 투영되길 원하시나요?
A: 구조적 안점감: 단단한 블록이 쌓여 만들어진 듯한 튼튼한 기초

Q3. 이 AI 코파일럿은 사용자에게 어떤 '파트너'로 인식되길 바라시나요?
A: 냉철한 분석가: 데이터와 논리로 브랜드를 진단하는 AI 전문가

아래 JSON 형식으로만 초안 후보 4개를 응답하세요:

{
  "candidates": [
    {
      "id": 1,
      "brand_name": "브랜드명 (국문, 2~20글자)",
      "brand_name_en": "영문명",
      "name_meaning": "이름의 의미와 어감 설명 (50자 이내)",
      "slogan": "핵심 슬로건 (국문, 15자 이내)",
      "story_summary": "브랜드 탄생 서사 요약 (100자 이내)",
      "seed_color": "#XXXXXX",
      "seed_color_reason": "이 컬러를 선택한 이유 (30자 이내)"
    },
    { "id": 2, "brand_name": "...", "brand_name_en": "...", "name_meaning": "...", "slogan": "...", "story_summary": "...", "seed_color": "...", "seed_co

In [7]:
raw_text_1 = bd.request_gemini_api(prompt_1, SYSTEM_PROMPT)
parsed_response_1 = bd.parse_ai_response(raw_text_1)


----------------------------------------------------------------------------------------------------
💬 AI에 요청 중입니다... 잠시만 기다려주세요.
----------------------------------------------------------------------------------------------------

✅ AI 응답 수신 완료!
응답 텍스트: {
  "candidates": [
    {
      "id": 1,
      "brand_name": "아키그램",
      "brand_name_en": "Archigr...
✅ JSON 파싱 완료!
파싱된 JSON: {"candidates": [{"id": 1, "brand_name": "\uc544\ud0a4\uadf8\ub7a8", "brand_name_en": "Archigram", "name_meaning": "\uac74\ucd95(Architecture)\uacfc \ub3c4\ud45c(Diagram)\uc758 \ud569\uc131\uc5b4", "slogan": "\ub17c\ub9ac\ub85c \uc313\uc544 \uc62c\ub9b0 \ube0c\ub79c\ub4dc\uc758 \uace8\uc870", "story_summary": "\ube0c\ub79c\ub529\uc744 \uac10\uc758 \uc601\uc5ed\uc774 \uc544\ub2cc \uac74\ucd95\uc801 \uc124\uacc4\uc758 \uc601\uc5ed\uc73c\ub85c \uc804\ud658\ud558\uc5ec, \ub370\uc774\ud130 \uae30\ubc18\uc758 \ub2e8\ub2e8\ud55c \uae30\ucd08\ub97c \uc81c\uc548\ud569\ub2c8\ub2e4.", "seed_color": "#0047AB", "seed_color

In [8]:
candidates = parsed_response_1.get("candidates", [])
num_candidates = len(candidates)

# 3. 통합 함수(select_item)를 활용하여 선택 진행
name_idx = bd.select_item(
    "1. 브랜드 이름 후보", 
    candidates, 
    lambda c: f"{c['brand_name']} ({c['brand_name_en']})", 
    "▶ 마음에 드는 브랜드 이름의 번호를 선택하세요: "
)

meaning_idx = bd.select_item(
    "2. 네이밍 의미 후보", 
    candidates, 
    lambda c: c['name_meaning'], 
    "▶ 마음에 드는 의미의 번호를 선택하세요: "
)

slogan_idx = bd.select_item(
    "3. 핵심 슬로건 후보", 
    candidates, 
    lambda c: c['slogan'], 
    "▶ 마음에 드는 슬로건의 번호를 선택하세요: "
)

story_idx = bd.select_item(
    "4. 스토리 요약 후보", 
    candidates, 
    lambda c: c['story_summary'], 
    "▶ 마음에 드는 스토리의 번호를 선택하세요: "
)

color_idx = bd.select_item(
    "5. 브랜드 컬러 후보", 
    candidates, 
    lambda c: f"{c['seed_color']} ({c['seed_color_reason']})", 
    "▶ 마음에 드는 컬러의 번호를 선택하세요: "
)

# 최종 선택 결과 조합 딕셔너리 생성
brand_info = {
    "brand_name": candidates[name_idx]["brand_name"],
    "brand_name_en": candidates[name_idx]["brand_name_en"],
    "name_meaning": candidates[meaning_idx]["name_meaning"],
    "slogan": candidates[slogan_idx]["slogan"],
    "story_summary": candidates[story_idx]["story_summary"],
    "seed_color": candidates[color_idx]["seed_color"],
    "seed_color_reason": candidates[color_idx]["seed_color_reason"]
}


[1. 브랜드 이름 후보]
  1. 아키그램 (Archigram)
  2. 그리드온 (Gridon)
  3. 로직셀 (Logicsel)
  4. 시스테마 (Systema)

[2. 네이밍 의미 후보]
  1. 건축(Architecture)과 도표(Diagram)의 합성어
  2. 기하학적 격자(Grid) 위에 브랜드를 바로 세움
  3. 논리(Logic)와 브랜드의 최소 단위(Cell)의 결합
  4. 체계와 질서를 뜻하는 라틴어 기반의 네이밍

[3. 핵심 슬로건 후보]
  1. 논리로 쌓아 올린 브랜드의 골조
  2. 오차 없는 정교한 브랜드 설계
  3. 데이터로 증명하는 브랜딩의 정석
  4. 브랜딩을 시스템으로 가동하다

[4. 스토리 요약 후보]
  1. 브랜딩을 감의 영역이 아닌 건축적 설계의 영역으로 전환하여, 데이터 기반의 단단한 기초를 제안합니다.
  2. 모든 디자인 요소를 기하학적 질서 속에 배치하여, 초기 창업자에게 완벽한 구조적 안정감을 제공합니다.
  3. 흩어진 아이디어를 냉철하게 분석하고 논리적 세포 단위로 재구성하여 설득력 있는 브랜드 결과물을 도출합니다.
  4. 개인의 취향을 넘어 기계적일 만큼 정확한 프로세스를 통해, 누구나 신뢰할 수 있는 브랜드 규격화 시스템을 지향합니다.

[5. 브랜드 컬러 후보]
  1. #0047AB (기술적 신뢰감과 전문성을 상징하는 코발트 블루)
  2. #1A1A1A (기계적인 정밀함과 현대적인 감각의 그라파이트 블랙)
  3. #007F5F (분석적인 안정감과 성장을 의미하는 딥 에메랄드)
  4. #4A4A4A (단단한 금속의 질감을 닮은 냉철한 스틸 그레이)


In [9]:
brand_info

{'brand_name': '시스테마',
 'brand_name_en': 'Systema',
 'name_meaning': '체계와 질서를 뜻하는 라틴어 기반의 네이밍',
 'slogan': '데이터로 증명하는 브랜딩의 정석',
 'story_summary': '흩어진 아이디어를 냉철하게 분석하고 논리적 세포 단위로 재구성하여 설득력 있는 브랜드 결과물을 도출합니다.',
 'seed_color': '#4A4A4A',
 'seed_color_reason': '단단한 금속의 질감을 닮은 냉철한 스틸 그레이'}